# 01 — Core Stop Event Table Pipeline

This notebook runs the full preprocessing pipeline and produces the core stop event table.

**Pipeline steps:**
1. Load raw DVB vehicle position data and timetable
2. Detect stop events (`detector.py`)
3. Expand timetable and compute delay, dwell_time, travel_time (`timetable.py`)
4. Add binary features: `is_peak_hour`, `is_workday`, `has_traffic_signal` (`feature_builder.py`)
5. Save core stop event table

## 0. Imports and paths

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import polars as pl

# Add project root to path so pipeline modules can be imported
sys.path.append("..")

from pipeline.detector import detect_stop_events
from pipeline.timetable import expand_timetable, match_and_compute_delay
from pipeline.feature_builder import add_binary_features

In [3]:
# ── Data paths ────────────────────────────────────────────────
RAW_VEHICLE_PATH   = "../data/regular_linie_week.csv"
RAW_TIMETABLE_PATH = "../data/timetable_trips_2025_07_22.csv"
OUTPUT_PATH        = "../data/processed/core_stop_events.parquet"

## 1. Load raw data

In [4]:
print("Loading raw vehicle position data...")
raw_df = (
    pl.read_csv(
        RAW_VEHICLE_PATH,
        schema_overrides={"linie_text": pl.Utf8}
    )
    .with_columns(
        pl.col("tst_iso").str.to_datetime(format="%Y-%m-%dT%H:%M:%S%.f%z")
    )
    .sort(["fzg_id", "tst_iso"])
    .with_columns(
        pl.int_range(pl.len()).over("fzg_id").alias("row_idx")
    )
)

print(f"Rows:       {len(raw_df):,}")
print(f"Vehicles:   {raw_df['fzg_id'].n_unique():,}")
print(f"Date range: {raw_df['tst_iso'].min()} → {raw_df['tst_iso'].max()}")
raw_df.head(3)

Loading raw vehicle position data...
Rows:       10,869,683
Vehicles:   538
Date range: 2025-07-28 00:00:00.515800+00:00 → 2025-08-03 23:59:59.146527+00:00


topic,tst,tst_iso,qos,retain,payloadlen,besetztgrad,distanz,fahrt_id,fzg_id,kurs,lage,linie,linie_text,ort_nr_start,ort_nr_start_vvo,ort_nr_ziel,ort_nr_ziel_vvo,pos_lat,pos_lon,status,status_text,tuerkriterium,zeitstempel,zieltext,row_idx
str,str,"datetime[μs, UTC]",i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,i64,i64,i64,i64,f64,f64,i64,str,bool,i64,str,i64
"""dvb/itcs/vehicleData/6/positio…","""2025-07-30T00:22:42.626209+020…",2025-07-29 22:22:42.626209 UTC,0,0,349,-1,0,-1,6,1,0,59,null,-1,-1,-1,-1,51.044835,13.653444,1,"""ohneFahrt""",false,0,null,0
"""dvb/itcs/vehicleData/6/positio…","""2025-07-30T00:22:45.626898+020…",2025-07-29 22:22:45.626898 UTC,0,0,349,-1,0,-1,6,1,0,59,null,-1,-1,-1,-1,51.044835,13.653444,1,"""ohneFahrt""",false,0,null,1
"""dvb/itcs/vehicleData/6/positio…","""2025-07-30T00:22:46.625247+020…",2025-07-29 22:22:46.625247 UTC,0,0,349,-1,0,-1,6,1,0,59,null,-1,-1,-1,-1,51.044835,13.653444,1,"""ohneFahrt""",false,0,null,2


In [5]:
print("Loading raw timetable...")
raw_tt = pl.read_csv(
    RAW_TIMETABLE_PATH,
    infer_schema_length=10000
)

print(f"Rows:          {len(raw_tt):,}")
print(f"Unique trips:  {raw_tt['fahrt_id'].n_unique():,}")
raw_tt.head(3)

Loading raw timetable...
Rows:          263,865
Unique trips:  97,982


topic,tst,tst_iso,qos,retain,payloadlen,fahrt_id,linie,ort_nr_abfahrt,route_nr,segmente,umlauf_id,wendezeit,zp_abfahrt
str,str,str,i64,i64,i64,f64,f64,f64,f64,str,f64,f64,f64
"""dvb/itcs/timetable/trips/13046…","""2025-07-22T13:25:52.616819+020…","""2025-07-22T11:25:52.616819+00:…",0,1,153,1.3046122e7,4.0,151401.0,254.0,"""[]""",465.0,0.0,1.7532e9
"""dvb/itcs/timetable/trips/86181…","""2025-07-22T13:25:52.616860+020…","""2025-07-22T11:25:52.616860+00:…",0,1,1333,8.618122e6,9.0,183803.0,151.0,"""[{""lenkzeit"": 60, ""ort_nr"": 18…",1337.0,null,1.7532e9
"""dvb/itcs/timetable/trips/31154…","""2025-07-22T13:25:52.616990+020…","""2025-07-22T11:25:52.616990+00:…",0,1,2060,3.1154222e7,63.0,645961.0,342.0,"""[{""lenkzeit"": 120, ""ort_nr"": 6…",6303.0,null,1.7532e9


In [6]:
import json
sample = raw_tt.filter(pl.col("segmente") != "[]")["segmente"][0]
print(json.dumps(json.loads(sample), indent=2))

[
  {
    "lenkzeit": 60,
    "ort_nr": 183901
  },
  {
    "lenkzeit": 60,
    "ort_nr": 184001
  },
  {
    "lenkzeit": 120,
    "ort_nr": 184101
  },
  {
    "lenkzeit": 60,
    "ort_nr": 183601
  },
  {
    "lenkzeit": 60,
    "ort_nr": 184301
  },
  {
    "lenkzeit": 60,
    "ort_nr": 184401
  },
  {
    "lenkzeit": 60,
    "ort_nr": 184501
  },
  {
    "lenkzeit": 60,
    "ort_nr": 184601
  },
  {
    "lenkzeit": 60,
    "ort_nr": 184701
  },
  {
    "lenkzeit": 60,
    "ort_nr": 184801
  },
  {
    "lenkzeit": 60,
    "ort_nr": 184901
  },
  {
    "lenkzeit": 120,
    "ort_nr": 185001
  },
  {
    "lenkzeit": 60,
    "ort_nr": 185301
  },
  {
    "lenkzeit": 60,
    "ort_nr": 185401
  },
  {
    "lenkzeit": 120,
    "ort_nr": 185801
  },
  {
    "lenkzeit": 120,
    "ort_nr": 185601
  },
  {
    "lenkzeit": 60,
    "ort_nr": 185701
  },
  {
    "lenkzeit": 60,
    "ort_nr": 192201
  },
  {
    "lenkzeit": 60,
    "ort_nr": 192203
  },
  {
    "lenkzeit": 60,
    "ort_nr": 191402

## 2. Detect stop events

In [7]:
print("Detecting stop events...")
stop_events = detect_stop_events(raw_df)

print(f"Stop events detected: {len(stop_events):,}")
print("\nstop_status distribution:")
print(
    stop_events
    .group_by("stop_status")
    .len()
    .rename({"len": "count"})
    .with_columns((pl.col("count") / len(stop_events) * 100).round(1).alias("%"))
    .sort("count", descending=True)
)
stop_events.head(3)

Detecting stop events...
Stop events detected: 1,021,431

stop_status distribution:
shape: (3, 3)
┌─────────────┬────────┬──────┐
│ stop_status ┆ count  ┆ %    │
│ ---         ┆ ---    ┆ ---  │
│ str         ┆ u32    ┆ f64  │
╞═════════════╪════════╪══════╡
│ normal      ┆ 840594 ┆ 82.3 │
│ no_door     ┆ 178212 ┆ 17.4 │
│ multi_door  ┆ 2625   ┆ 0.3  │
└─────────────┴────────┴──────┘


fzg_id,drop_row_idx,drop_time,linie,window_lo,window_hi,arrival_time,departure_time,door_open_count,door_close_count,door_near_drop,is_true_multi_door,delta_at_drop,min_distanz,max_distanz,stop_status
i64,i64,"datetime[μs, UTC]",i64,i64,i64,"datetime[μs, UTC]","datetime[μs, UTC]",i64,i64,bool,bool,i64,i64,i64,str
151,106,2025-07-29 02:23:54.604010 UTC,4,102,110,2025-07-29 02:23:54.604010 UTC,2025-07-29 02:23:54.604010 UTC,1,1,true,false,-603,15,618,"""normal"""
151,116,2025-07-29 02:25:44.604703 UTC,4,112,120,2025-07-29 02:25:44.604703 UTC,2025-07-29 02:25:44.604703 UTC,0,0,false,false,-586,20,606,"""no_door"""
151,124,2025-07-29 02:26:37.622760 UTC,4,120,128,2025-07-29 02:26:37.622760 UTC,2025-07-29 02:26:37.622760 UTC,0,0,false,false,-262,60,389,"""no_door"""


In [11]:
print("Expanding timetable...")
timetable = expand_timetable(raw_tt)

print(f"Stop-level rows:     {len(timetable):,}")
print(f"Unique trips:        {timetable['fahrt_id'].n_unique():,}")
print(f"Unique stops (ort_nr): {timetable['ort_nr'].n_unique():,}")
timetable.head(3)

Expanding timetable...
Stop-level rows:     2,381,003
Unique trips:        97,591
Unique stops (ort_nr): 2,006


fahrt_id,stop_index,ort_nr,scheduled_arrival_unix,scheduled_arrival_time
i64,i64,i64,f64,datetime[μs]
11746102,0,155701,1.7541e9,2025-08-02 11:52:00
11746102,1,155801,1.7541e9,2025-08-02 11:53:00
11746102,2,156101,1.7541e9,2025-08-02 11:54:00


In [12]:
print("Expanding timetable...")
timetable = expand_timetable(raw_tt)

print(f"Stop-level rows:     {len(timetable):,}")
print(f"Unique trips:        {timetable['fahrt_id'].n_unique():,}")
print(f"Unique stops (ort_nr): {timetable['ort_nr'].n_unique():,}")
timetable.head(3)

Expanding timetable...
Stop-level rows:     2,381,003
Unique trips:        97,591
Unique stops (ort_nr): 2,006


fahrt_id,stop_index,ort_nr,scheduled_arrival_unix,scheduled_arrival_time
i64,i64,i64,f64,datetime[μs]
18411203,0,631461,1.7542e9,2025-08-03 07:18:00
18411203,1,631361,1.7542e9,2025-08-03 07:20:00
18411203,2,631861,1.7542e9,2025-08-03 07:21:00


In [13]:
print("Matching stop events to timetable...")
matched = match_and_compute_delay(stop_events, raw_df, timetable)

total      = len(stop_events)
matched_n  = len(matched)

print(f"Total stop events:    {total:,}")
print(f"Successfully matched: {matched_n:,}  ({matched_n/total*100:.1f}%)")
print(f"Unmatched:            {total-matched_n:,}  ({(total-matched_n)/total*100:.1f}%)")
matched.head(3)

Matching stop events to timetable...
Total stop events:    1,021,431
Successfully matched: 994,117  (97.3%)
Unmatched:            27,314  (2.7%)


fzg_id,drop_row_idx,arrival_time,departure_time,linie,fahrt_id,ort_nr_start,stop_index,stop_status,scheduled_arrival_time,delay_calculated_sec,delay_recorded_sec,dwell_time,travel_time,besetztgrad
i64,i64,"datetime[μs, UTC]","datetime[μs, UTC]",i64,i64,i64,i64,str,"datetime[μs, UTC]",f64,i64,f64,f64,i64
151,106,2025-07-29 02:23:54.604010 UTC,2025-07-29 02:23:54.604010 UTC,4,5924129,184104,0,"""normal""",2025-07-29 02:22:00 UTC,114.60401,-10,0.0,null,1
151,116,2025-07-29 02:25:44.604703 UTC,2025-07-29 02:25:44.604703 UTC,4,5924129,182302,1,"""no_door""",2025-07-29 02:24:00 UTC,104.604703,-20,0.0,110.000693,0
151,124,2025-07-29 02:26:37.622760 UTC,2025-07-29 02:26:37.622760 UTC,4,5924129,182402,2,"""no_door""",2025-07-29 02:26:00 UTC,37.62276,-20,0.0,53.018057,0


In [14]:
# Sanity check on computed fields
for field in ["delay_calculated_sec", "dwell_time", "travel_time"]:
    print(f"\n{field} stats (seconds):")
    print(matched.select(field).describe())


delay_calculated_sec stats (seconds):
shape: (9, 2)
┌────────────┬──────────────────────┐
│ statistic  ┆ delay_calculated_sec │
│ ---        ┆ ---                  │
│ str        ┆ f64                  │
╞════════════╪══════════════════════╡
│ count      ┆ 994117.0             │
│ null_count ┆ 0.0                  │
│ mean       ┆ 164.346216           │
│ std        ┆ 137.528108           │
│ min        ┆ -8042.994951         │
│ 25%        ┆ 81.805588            │
│ 50%        ┆ 135.291498           │
│ 75%        ┆ 213.106284           │
│ max        ┆ 4934.212474          │
└────────────┴──────────────────────┘

dwell_time stats (seconds):
shape: (9, 2)
┌────────────┬───────────────┐
│ statistic  ┆ dwell_time    │
│ ---        ┆ ---           │
│ str        ┆ f64           │
╞════════════╪═══════════════╡
│ count      ┆ 994117.0      │
│ null_count ┆ 0.0           │
│ mean       ┆ 6.010609      │
│ std        ┆ 265.368475    │
│ min        ┆ -2180.270519  │
│ 25%        ┆ 0.0      

## 4. Add binary features

In [15]:
print("Adding binary features...")
core_table = add_binary_features(matched)

print("is_peak_hour distribution:")
print(core_table.group_by("is_peak_hour").len().sort("is_peak_hour"))

print("\nis_workday distribution:")
print(core_table.group_by("is_workday").len().sort("is_workday"))

print("\nhas_traffic_signal: placeholder (all null — awaiting infrastructure data)")

Adding binary features...
is_peak_hour distribution:
shape: (2, 2)
┌──────────────┬────────┐
│ is_peak_hour ┆ len    │
│ ---          ┆ ---    │
│ i8           ┆ u32    │
╞══════════════╪════════╡
│ 0            ┆ 734551 │
│ 1            ┆ 259566 │
└──────────────┴────────┘

is_workday distribution:
shape: (2, 2)
┌────────────┬────────┐
│ is_workday ┆ len    │
│ ---        ┆ ---    │
│ i8         ┆ u32    │
╞════════════╪════════╡
│ 0          ┆ 398167 │
│ 1          ┆ 595950 │
└────────────┴────────┘

has_traffic_signal: placeholder (all null — awaiting infrastructure data)


## 5. Final schema check and save

In [16]:
# Final schema — should match the Core Stop Event Table in the data structure plan
print(f"Total rows: {len(core_table):,}\n")
print(f"{'Column':<30} {'Type':<20} {'Nulls':>10}")
print("-" * 62)
for col, dtype in zip(core_table.columns, core_table.dtypes):
    null_count = core_table[col].null_count()
    null_pct   = null_count / len(core_table) * 100
    print(f"{col:<30} {str(dtype):<20} {null_count:>6,} ({null_pct:.1f}%)")

Total rows: 994,117

Column                         Type                      Nulls
--------------------------------------------------------------
fzg_id                         Int64                     0 (0.0%)
drop_row_idx                   Int64                     0 (0.0%)
arrival_time                   Datetime(time_unit='us', time_zone='UTC')      0 (0.0%)
departure_time                 Datetime(time_unit='us', time_zone='UTC')      0 (0.0%)
linie                          Int64                     0 (0.0%)
fahrt_id                       Int64                     0 (0.0%)
ort_nr_start                   Int64                     0 (0.0%)
stop_index                     Int64                     0 (0.0%)
stop_status                    String                    0 (0.0%)
scheduled_arrival_time         Datetime(time_unit='us', time_zone='UTC')      0 (0.0%)
delay_calculated_sec           Float64                   0 (0.0%)
delay_recorded_sec             Int64                     0 (0.0%

In [23]:
import os
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

print(f"Saving to {OUTPUT_PATH} ...")
core_table.write_parquet(OUTPUT_PATH)
print(f"Done. {len(core_table):,} rows saved.")

Saving to ../data/processed/core_stop_events.parquet ...
Done. 994,117 rows saved.
